# E2E 01 - Builder Graph Smoke Test

Questo notebook testa l'intero flusso del **Builder Graph**: ingestione documento, parsing DDL, mapping, generazione Cypher e upsert su Neo4j.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import fitz

from src.graph.builder_graph import run_builder
from src.graph.neo4j_client import Neo4jClient


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml non trovato risalendo dalla cwd")


repo_root = find_repo_root(Path.cwd())
repo_root

/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')


PosixPath('/home/marcantoniolopez/Documenti/github/thesis')

In [2]:
fixtures_dir = repo_root / "tests" / "fixtures"
docs_dir = fixtures_dir / "sample_docs"
ddl_file = fixtures_dir / "sample_ddl" / "simple_schema.sql"
generated_dir = repo_root / "notebooks" / "e2e-tests" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)

pdf_path = generated_dir / "business_context_e2e.pdf"
text_sources = [docs_dir / "business_glossary.txt", docs_dir / "data_dictionary.txt"]

combined_text = []
for src in text_sources:
    combined_text.append(f"# Source: {src.name}\n")
    combined_text.append(src.read_text(encoding="utf-8"))

doc = fitz.open()
page = doc.new_page()
text = "\n\n".join(combined_text)
page.insert_textbox(
    fitz.Rect(40, 40, 555, 800),
    text,
    fontsize=10,
    fontname="helv",
    align=0,
)
doc.save(pdf_path)
doc.close()

print(f"PDF creato: {pdf_path}")
print(f"DDL usato:   {ddl_file}")

PDF creato: /home/marcantoniolopez/Documenti/github/thesis/notebooks/e2e-tests/generated/business_context_e2e.pdf
DDL usato:   /home/marcantoniolopez/Documenti/github/thesis/tests/fixtures/sample_ddl/simple_schema.sql


In [3]:
with Neo4jClient() as neo4j:
    ok = neo4j.test_connection()

assert ok, "Connessione Neo4j fallita. Verifica NEO4J_URI/USER/PASSWORD nel tuo .env"
print("Neo4j raggiungibile")

ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 111] Connection refused)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)

In [ ]:
final_state = run_builder(
    raw_documents=[str(pdf_path)],
    ddl_paths=[str(ddl_file)],
    production=False,
)

summary = {
    "triplets": len(final_state.get("triplets", [])),
    "entities": len(final_state.get("entities", [])),
    "tables": len(final_state.get("tables", [])),
    "enriched_tables": len(final_state.get("enriched_tables", [])),
    "completed_tables": len(final_state.get("completed_tables", [])),
    "cypher_failed": final_state.get("cypher_failed", False),
}

print(json.dumps(summary, indent=2))

In [ ]:
assert len(final_state.get("triplets", [])) >= 0
assert len(final_state.get("tables", [])) > 0, "Nessuna tabella estratta dal DDL"
assert len(final_state.get("completed_tables", [])) > 0, "Nessuna tabella completata nel builder"
assert final_state.get("cypher_failed", False) is False, "Cypher healing fallito definitivamente"
print("Builder Graph E2E smoke test completato con successo")